In [23]:
import numpy as np
import pandas as pd
import joblib

# Load model and encoder
crop_model = joblib.load("..\models\crop_recommendation_model.pkl")
label_encoder = joblib.load("..\models\label_encoder.pkl")
feature_columns = joblib.load("..\models\crop_feature_columns.pkl")

print("Model Loaded Successfully")


Model Loaded Successfully


In [24]:
sample_input = pd.DataFrame([{
    "N": 90,
    "P": 42,
    "K": 43,
    "temperature": 20.8,
    "humidity": 82.0,
    "ph": 6.5,
    "rainfall": 202.0
}])

In [25]:
sample_input = sample_input[feature_columns]

In [26]:
probs = crop_model.predict_proba(sample_input)

print("Probabilities shape:", probs.shape)

Probabilities shape: (1, 22)


In [27]:
top_n = 3

top_indices = np.argsort(probs[0])[-top_n:][::-1]

top_crops = label_encoder.inverse_transform(top_indices)
top_probs = probs[0][top_indices]

In [28]:
print("Top 3 Recommended Crops:\n")

for i, (crop, prob) in enumerate(zip(top_crops, top_probs), start=1):
    print(f"{i}. {crop} → {prob*100:.2f}% suitability")

Top 3 Recommended Crops:

1. rice → 99.75% suitability
2. jute → 0.07% suitability
3. coffee → 0.02% suitability


In [29]:
from xgboost import XGBRegressor

yield_model = XGBRegressor()
yield_model.load_model("..\models\yield_model.json")

print("Model loaded successfully")

Model loaded successfully


In [31]:
import numpy as np
import pandas as pd
import joblib
from xgboost import XGBRegressor

# Load crop model
from xgboost import XGBClassifier

crop_model = XGBClassifier()
crop_model.load_model("..\models\crop_model.json")
label_encoder = joblib.load("..\models\label_encoder.pkl")
crop_columns = joblib.load("..\models\crop_feature_columns.pkl")

# Load yield model
yield_model = XGBRegressor()
yield_model.load_model("..\models\yield_model.json")

yield_columns = joblib.load("..\models\yield_feature_columns.pkl")

print("All models loaded")

All models loaded


In [32]:
crop_input = pd.DataFrame([{
    "N": 90,
    "P": 42,
    "K": 43,
    "temperature": 20.8,
    "humidity": 82.0,
    "ph": 6.5,
    "rainfall": 202.0
}])

crop_input = crop_input[crop_columns]

In [33]:
probs = crop_model.predict_proba(crop_input)

top_n = 3
top_indices = np.argsort(probs[0])[-top_n:][::-1]

top_crops = label_encoder.inverse_transform(top_indices)
top_probs = probs[0][top_indices]

print("Top Crops:")
for crop, prob in zip(top_crops, top_probs):
    print(f"{crop} → {prob*100:.2f}%")

Top Crops:
rice → 99.75%
jute → 0.07%
coffee → 0.02%


In [39]:
results = []

for crop in top_crops:
    
    # Create empty row
    row = pd.DataFrame(columns=yield_columns)
    row.loc[0] = 0
    
    # Fill base values
    row["Area"] = 2.0
    row["Annual_Rainfall"] = crop_input["rainfall"].values[0]
    row["Fertilizer"] = 120
    row["Pesticide"] = 30
    row["Crop_Year"] = 2020
    
    # Set state & season
    row["State_Tamil_Nadu"] = 1
    row["Season_Kharif"] = 1
    
    # Set crop column
    for col in row.columns:
        if col.startswith("Crop_") and crop.lower() in col.lower():
            row[col] = 1
    
    # Predict
    log_yield = yield_model.predict(row)
    yield_val = np.expm1(log_yield)[0]
    
    results.append((crop, yield_val))

In [40]:
print("\nFinal Recommendation with Yield:\n")

for (crop, prob), (_, y) in zip(zip(top_crops, top_probs), results):
    print(f"{crop} → {prob*100:.2f}% suitability | Yield: {y:.2f}")
    #print(row.loc[:, row.sum() != 0])


Final Recommendation with Yield:

rice → 99.75% suitability | Yield: 0.95
jute → 0.07% suitability | Yield: 0.48
coffee → 0.02% suitability | Yield: 0.30


These values are small because the model predicts log-transformed yield

In [42]:
def predict_crop_and_yield(input_dict):
    
    # Convert input
    crop_input = pd.DataFrame([input_dict])
    crop_input = crop_input[crop_columns]

    # Predict top crops
    probs = crop_model.predict_proba(crop_input)
    top_indices = np.argsort(probs[0])[-3:][::-1]

    top_crops = label_encoder.inverse_transform(top_indices)
    top_probs = probs[0][top_indices]

    results = []

    for crop, prob in zip(top_crops, top_probs):

        row = pd.DataFrame(columns=yield_columns)
        row.loc[0] = 0

        # Numeric features
        row["Area"] = 2.0
        row["Annual_Rainfall"] = crop_input["rainfall"].values[0]
        row["Fertilizer"] = 120
        row["Pesticide"] = 30
        row["Crop_Year"] = 2020

        # State
        for col in row.columns:
            if col.startswith("State_") and "Tamil_Nadu" in col:
                row[col] = 1

        # Season
        for col in row.columns:
            if col.startswith("Season_") and "Kharif" in col:
                row[col] = 1

        # Crop encoding
        for col in row.columns:
            if col.startswith("Crop_") and crop.lower() in col.lower():
                row[col] = 1

        # Predict yield
        log_yield = yield_model.predict(row)
        yield_val = np.expm1(log_yield)[0]

        results.append({
            "crop": crop,
            "probability": prob,
            "yield": yield_val
        })

    return results

In [43]:
input_data = {
    "N": 90,
    "P": 42,
    "K": 43,
    "temperature": 20.8,
    "humidity": 82.0,
    "ph": 6.5,
    "rainfall": 202.0
}

output = predict_crop_and_yield(input_data)

for item in output:
    print(item)

{'crop': 'rice', 'probability': np.float32(0.99747926), 'yield': np.float32(0.9455446)}
{'crop': 'jute', 'probability': np.float32(0.0007139892), 'yield': np.float32(0.47934783)}
{'crop': 'coffee', 'probability': np.float32(0.0002021918), 'yield': np.float32(0.29581517)}
